In [ ]:
import json
from pathlib import Path
from typing import Iterable, Union

import pandas as pd


def load_runs_into_df(run_dir: Union[str, Path], recursive: bool = True) -> pd.DataFrame:
    """Load Slay the Spire 2 .run JSON files into a metadata DataFrame.

    Args:
        run_dir: Directory containing .run files.
        recursive: Whether to search subdirectories recursively.

    Returns:
        pd.DataFrame with one row per run file and key metadata columns.
    """
    run_dir = Path(run_dir).expanduser()
    if not run_dir.exists():
        raise FileNotFoundError(f"Run directory does not exist: {run_dir}")

    pattern = "**/*.run" if recursive else "*.run"
    run_files: Iterable[Path] = sorted(run_dir.glob(pattern))

    rows = []
    for run_file in run_files:
        try:
            with run_file.open("r", encoding="utf-8") as f:
                data = json.load(f)
        except Exception as exc:
            rows.append(
                {
                    "file_path": str(run_file),
                    "file_name": run_file.name,
                    "parse_error": str(exc),
                }
            )
            continue

        players = data.get("players", []) or []
        first_player = players[0] if players else {}

        rows.append(
            {
                "file_path": str(run_file),
                "file_name": run_file.name,
                "seed": data.get("seed"),
                "start_time": data.get("start_time"),
                "run_time_seconds": data.get("run_time"),
                "win": data.get("win"),
                "was_abandoned": data.get("was_abandoned"),
                "ascension": data.get("ascension"),
                "build_id": data.get("build_id"),
                "platform_type": data.get("platform_type"),
                "schema_version": data.get("schema_version"),
                "character": first_player.get("character"),
                "deck_size": len(first_player.get("deck", []) or []),
                "relic_count": len(first_player.get("relics", []) or []),
                "killed_by_encounter": data.get("killed_by_encounter"),
                "killed_by_event": data.get("killed_by_event"),
                "parse_error": None,
            }
        )

    df = pd.DataFrame(rows)

    if not df.empty and "start_time" in df.columns:
        df["start_time_dt"] = pd.to_datetime(df["start_time"], unit="s", errors="coerce")

    return df


# Example:
df = load_runs_into_df('/Users/jzd361/Library/Application Support/SlayTheSpire2/steam/76561198187362545/profile1/saves/history')
df.head()


,file_path,file_name,seed,start_time,run_time_seconds,win,was_abandoned,ascension,game_mode,build_id,platform_type,schema_version,character,deck_size,relic_count,killed_by_encounter,killed_by_event,parse_error,start_time_dt
0,/Users/jzd361/Library/Application Support/Slay...,1772746838.run,ZZHDCKMPCE,1772746838,4546,True,False,0,standard,v0.98.0,steam,8,CHARACTER.IRONCLAD,30,16,NONE.NONE,NONE.NONE,None,2026-03-05 21:40:38
1,/Users/jzd361/Library/Application Support/Slay...,1772751643.run,P7HB90AZ0D,1772751643,3766,True,False,0,standard,v0.98.0,steam,8,CHARACTER.SILENT,39,14,NONE.NONE,NONE.NONE,None,2026-03-05 23:00:43
2,/Users/jzd361/Library/Application Support/Slay...,1772755633.run,6UQ18BGC2W,1772755633,2113,False,False,0,standard,v0.98.0,steam,8,CHARACTER.REGENT,18,8,ENCOUNTER.HUNTER_KILLER_NORMAL,NONE.NONE,None,2026-03-06 00:07:13
3,/Users/jzd361/Library/Application Support/Slay...,1772757850.run,22LN6LLFA9,1772757850,711,False,False,0,standard,v0.98.0,steam,8,CHARACTER.NECROBINDER,19,4,ENCOUNTER.SKULKING_COLONY_ELITE,NONE.NONE,None,2026-03-06 00:44:10
4,/Users/jzd361/Library/Application Support/Slay...,1772758599.run,ESQ8T5SSQ8,1772758599,2911,True,False,0,standard,v0.98.0,steam,8,CHARACTER.DEFECT,31,13,NONE.NONE,NONE.NONE,None,2026-03-06 00:56:39


In [ ]:
def _safe_join(items, sep=" | "):
    return sep.join(str(x) for x in items) if items else ""


def _extract_act_enemies(map_point_history):
    act_enemies = {}
    for act_idx, act_nodes in enumerate(map_point_history or [], start=1):
        enemies = []
        for node in act_nodes or []:
            for room in node.get("rooms", []) or []:
                model_id = room.get("model_id")
                if model_id:
                    enemies.append(model_id)
        act_enemies[act_idx] = enemies
    return act_enemies


def load_runs_into_df(run_dir: Union[str, Path], recursive: bool = True) -> pd.DataFrame:
    """Load .run files into a metadata-rich dataframe (one row per run)."""
    run_dir = Path(run_dir).expanduser()
    if not run_dir.exists():
        raise FileNotFoundError(f"Run directory does not exist: {run_dir}")

    pattern = "**/*.run" if recursive else "*.run"
    run_files: Iterable[Path] = sorted(run_dir.glob(pattern))

    rows = []
    for run_file in run_files:
        try:
            with run_file.open("r", encoding="utf-8") as f:
                data = json.load(f)
        except Exception as exc:
            rows.append({"file_path": str(run_file), "file_name": run_file.name, "parse_error": str(exc)})
            continue

        players = data.get("players", []) or []
        first_player = players[0] if players else {}

        deck = first_player.get("deck", []) or []
        relics = first_player.get("relics", []) or []
        acts = data.get("acts", []) or []
        map_history = data.get("map_point_history", []) or []

        deck_ids = [c.get("id") for c in deck if c.get("id")]
        relic_ids = [r.get("id") for r in relics if r.get("id")]
        upgraded_cards = [c.get("id") for c in deck if c.get("id") and (c.get("current_upgrade_level", 0) or 0) > 0]

        act_enemies = _extract_act_enemies(map_history)
        all_enemies = [enemy for enemies in act_enemies.values() for enemy in enemies]

        row = {
            "file_path": str(run_file),
            "file_name": run_file.name,
            "seed": data.get("seed"),
            "start_time": data.get("start_time"),
            "run_time_seconds": data.get("run_time"),
            "win": data.get("win"),
            "was_abandoned": data.get("was_abandoned"),
            "ascension": data.get("ascension"),
            "game_mode": data.get("game_mode"),
            "build_id": data.get("build_id"),
            "platform_type": data.get("platform_type"),
            "schema_version": data.get("schema_version"),
            "character": first_player.get("character"),
            "killed_by_encounter": data.get("killed_by_encounter"),
            "killed_by_event": data.get("killed_by_event"),
            "num_acts": len(acts),
            "acts": _safe_join(acts),
            "deck_size": len(deck_ids),
            "deck_cards": _safe_join(deck_ids),
            "unique_deck_cards": len(set(deck_ids)),
            "upgraded_cards": _safe_join(upgraded_cards),
            "num_upgraded_cards": len(upgraded_cards),
            "relic_count": len(relic_ids),
            "relics": _safe_join(relic_ids),
            "unique_relics": len(set(relic_ids)),
            "enemy_count": len(all_enemies),
            "enemies_encountered": _safe_join(all_enemies),
            "unique_enemy_count": len(set(all_enemies)),
            "unique_enemies": _safe_join(sorted(set(all_enemies))),
            "parse_error": None,
        }

        for act_idx, act_name in enumerate(acts, start=1):
            enemies_this_act = act_enemies.get(act_idx, [])
            row[f"act{act_idx}_name"] = act_name
            row[f"act{act_idx}_enemy_count"] = len(enemies_this_act)
            row[f"act{act_idx}_enemies"] = _safe_join(enemies_this_act)
            row[f"act{act_idx}_unique_enemies"] = _safe_join(sorted(set(enemies_this_act)))

        rows.append(row)

    df = pd.DataFrame(rows)
    if not df.empty and "start_time" in df.columns:
        df["start_time_dt"] = pd.to_datetime(df["start_time"], unit="s", errors="coerce")
    return df


# Re-run after defining this function:
# df = load_runs_into_df('/Users/jzd361/Library/Application Support/SlayTheSpire2/steam/76561198187362545/profile1/saves/history')
# df.head()


In [4]:
import json
from pprint import pprint

In [3]:
with open("1772746838.run", "rb") as f:
    data = json.load(f)

print(data)


{'acts': ['ACT.OVERGROWTH', 'ACT.HIVE', 'ACT.GLORY'], 'ascension': 0, 'build_id': 'v0.98.0', 'game_mode': 'standard', 'killed_by_encounter': 'NONE.NONE', 'killed_by_event': 'NONE.NONE', 'map_point_history': [[{'map_point_type': 'monster', 'player_stats': [{'card_choices': [{'card': {'id': 'CARD.SETUP_STRIKE'}, 'was_picked': False}, {'card': {'id': 'CARD.TREMBLE'}, 'was_picked': False}, {'card': {'id': 'CARD.BLOOD_WALL'}, 'was_picked': False}], 'current_gold': 110, 'current_hp': 80, 'damage_taken': 3, 'gold_gained': 11, 'gold_lost': 0, 'gold_spent': 0, 'gold_stolen': 0, 'hp_healed': 3, 'max_hp': 80, 'max_hp_gained': 0, 'max_hp_lost': 0, 'player_id': 1}], 'rooms': [{'model_id': 'ENCOUNTER.NIBBITS_WEAK', 'monster_ids': ['MONSTER.NIBBIT'], 'room_type': 'monster', 'turns_taken': 4}]}, {'map_point_type': 'monster', 'player_stats': [{'card_choices': [{'card': {'floor_added_to_deck': 2, 'id': 'CARD.INFLAME'}, 'was_picked': True}, {'card': {'id': 'CARD.BREAKTHROUGH'}, 'was_picked': False}, {'ca

In [5]:
pprint(data)

{'acts': ['ACT.OVERGROWTH', 'ACT.HIVE', 'ACT.GLORY'],
 'ascension': 0,
 'build_id': 'v0.98.0',
 'game_mode': 'standard',
 'killed_by_encounter': 'NONE.NONE',
 'killed_by_event': 'NONE.NONE',
 'map_point_history': [[{'map_point_type': 'monster',
                         'player_stats': [{'card_choices': [{'card': {'id': 'CARD.SETUP_STRIKE'},
                                                             'was_picked': False},
                                                            {'card': {'id': 'CARD.TREMBLE'},
                                                             'was_picked': False},
                                                            {'card': {'id': 'CARD.BLOOD_WALL'},
                                                             'was_picked': False}],
                                           'current_gold': 110,
                                           'current_hp': 80,
                                           'damage_taken': 3,
                               

In [ ]:
import pandas as pd

def load_runs_into_df(runs):
    df = pd.DataFrame(runs)
    return df

# df = load_runs_into_df(data)
# pprint(df)

ValueError: All arrays must be of the same length